In [1]:
import os
import sys

current_path = os.path.abspath(os.getcwd())
base_path = os.path.abspath(os.path.join(current_path, '../'))
sys.path.append(base_path)

import numpy as np
import matplotlib.pyplot as plt  
import pandas as pd
from tqdm import tqdm
import pandas as pd
from IPython.display import display

import project_images_loader as pil
import utils.util as util
import utils.functions as fc
import utils.nifti_functions as nfc
import utils.visualization as vis
from scipy import stats

np.set_printoptions(precision=4)

val_ids = ["08202"] + pil.get_matching_ids()[621:]


SERVER: False


# Initial metrics

In [2]:
features_loader_tissue_based = fc.load_module_from_path('features_loader_tissue_based', os.path.join(base_path, 'statistics/patients/longitudinal/tissue_based/features_loader.py'))
initial_longitudinal_loader = features_loader_tissue_based.FeaturesLoader(3, val_ids)
initial_longitudinal_data = initial_longitudinal_loader.get_df(metrics_name=['dice'], remove_metrics=False, return_patient_id=True)

def get_initial_metric(metric):
    _val_ids = [int(i) for i in val_ids]
    if metric == "ssim":
        _metric = pd.read_csv(f'{base_path}/statistics/patients/longitudinal/whole_brain_based/features/whole_brains_stats.csv')
        _metric = _metric[_metric['patient_id'].isin(_val_ids)]
        _metric = _metric['ssim']
    elif metric == "dice_csf":
        _metric = initial_longitudinal_data["1_dice"]
    elif metric == "dice_gm":
        _metric = initial_longitudinal_data["2_dice"]
    elif metric == "dice_wm":
        _metric = initial_longitudinal_data["3_dice"]
    return _metric

def get_initial_mean(metric):
    return get_initial_metric(metric).mean()


# Choose between methods

# ssim

In [3]:
def get_result_metric(approach_name, statistics_folder, metric):
    _metric = pd.read_csv(f'{current_path}/{approach_name}/images/{statistics_folder}/{metric}.csv')
    return _metric

def get_mean_methods(approach_name, statistics_folder, metric):
    return get_result_metric(approach_name, statistics_folder, metric).mean().iloc[1:]

metric = 'ssim'
mean_initial = get_initial_mean(metric)
mean_ssim_dfield_mean = get_mean_methods('dfield_mean', 'statistics', metric)
mean_ssim_dfield_similar_images = get_mean_methods('dfield_similar_images', 'statistics', metric)
mean_ssim_dfield_similar_images_registered = get_mean_methods('dfield_similar_images_registered', 'statistics', metric)
mean_ssim_dfield_similar_tissues = get_mean_methods('dfield_similar_tissues', 'statistics', metric)
mean_ssim_dfield_similar_tissues2 = get_mean_methods('dfield_similar_tissues_v2', 'statistics', metric)
mean_ssim_dfield_similar_patches = get_mean_methods('dfield_similar_patches', 'statistics', metric)
# mean_ssim_MPGAN_paper_16_seg_loss = get_mean_methods('MPGAN_paper_16_seg_loss', 'statistics', 'ssim')
# mean_ssim_MGAN_paper_16 = get_mean_methods('MGAN_paper_16', 'statistics', 'ssim')
# mean_ssim_styleGAN_paper_16 = get_mean_methods('StarGAN_V2_sagital', 'statistics', 'ssim')

print('INITIAL:\n', mean_initial)
print('\nmean_ssim_dfield_mean:\n', mean_ssim_dfield_mean)
print('\nmean_ssim_dfield_similar_images:\n', mean_ssim_dfield_similar_images)
print('\nmean_ssim_dfield_similar_images_registered:\n', mean_ssim_dfield_similar_images_registered)
print('\nmean_ssim_dfield_similar_tissues:\n', mean_ssim_dfield_similar_tissues)
print('\mean_ssim_dfield_similar_tissues2:\n', mean_ssim_dfield_similar_tissues2)
print('\nmean_ssim_dfield_similar_patches:\n', mean_ssim_dfield_similar_patches)
# print('\nmean_ssim_MPGAN_paper_16_seg_loss:\n', mean_ssim_MPGAN_paper_16_seg_loss)
# print('\nmean_ssim_MGAN_paper_16:\n', mean_ssim_MGAN_paper_16)
# print('\nmean_ssim_styleGAN_paper_16:\n', mean_ssim_styleGAN_paper_16)






INITIAL:
 0.9463590432890597

mean_ssim_dfield_mean:
 10     0.950641
100    0.953299
200    0.953384
300    0.953494
620    0.953516
dtype: float64

mean_ssim_dfield_similar_images:
 SSIM_5           0.953266
SSIM_10          0.954281
SSIM_100         0.954547
DICE_MEAN_5      0.953071
DICE_MEAN_10     0.954213
DICE_MEAN_100    0.954814
dtype: float64

mean_ssim_dfield_similar_images_registered:
 AFF_5         0.953117
BSP_FAST_5    0.953364
dtype: float64

mean_ssim_dfield_similar_tissues:
 3_10      0.954993
3_100     0.954993
96_10     0.955169
96_100    0.955251
dtype: float64
\mean_ssim_dfield_similar_tissues2:
 96_10    0.95466
dtype: float64

mean_ssim_dfield_similar_patches:
 3_10      0.955212
3_100     0.954985
96_10     0.955266
96_100    0.955034
dtype: float64

mean_ssim_MPGAN_paper_16_seg_loss:
 46    0.949630
48    0.952463
50    0.949980
52    0.956479
54    0.955259
56    0.953981
58    0.951348
60    0.954437
62    0.951090
64    0.952359
66    0.953012
68    0.95102

## dice scores

In [4]:
def get_mean_dices(approach_name, statistics_folder):
    _dice_csf = pd.read_csv(f'{current_path}/{approach_name}/images/{statistics_folder}/dice_csf.csv')
    _dice_gm = pd.read_csv(f'{current_path}/{approach_name}/images/{statistics_folder}/dice_gm.csv')
    _dice_wm = pd.read_csv(f'{current_path}/{approach_name}/images/{statistics_folder}/dice_wm.csv')
    _mean_csf = _dice_csf.mean().iloc[1:]
    _mean_gm = _dice_gm.mean().iloc[1:]
    _mean_wm = _dice_wm.mean().iloc[1:]

    means = pd.DataFrame({
        'mean_csf': _mean_csf,
        'mean_gm': _mean_gm,
        'mean_wm': _mean_wm
    })
    return means

mean_initial = pd.DataFrame(data=[[get_initial_mean("dice_csf"), get_initial_mean("dice_gm"), get_initial_mean("dice_wm")]], columns=['mean_csf', 'mean_gm', 'mean_wm'])
mean_ssim_dfield_mean = get_mean_dices('dfield_mean', 'statistics')
mean_ssim_dfield_similar_images = get_mean_dices('dfield_similar_images', 'statistics')
mean_ssim_dfield_similar_images_registered = get_mean_dices('dfield_similar_images_registered', 'statistics')
mean_ssim_dfield_similar_tissues = get_mean_dices('dfield_similar_tissues', 'statistics')
mean_ssim_dfield_similar_tissues2 = get_mean_dices('dfield_similar_tissues_v2', 'statistics')
mean_ssim_dfield_similar_patches = get_mean_dices('dfield_similar_patches', 'statistics')

print('INITIAL:\n', mean_initial)
print('\nmean_ssim_dfield_mean:\n', mean_ssim_dfield_mean)
print('\nmean_ssim_dfield_similar_images:\n', mean_ssim_dfield_similar_images)
print('\nmean_ssim_dfield_similar_images_registered:\n', mean_ssim_dfield_similar_images_registered)
print('\nmean_ssim_dfield_similar_tissues:\n', mean_ssim_dfield_similar_tissues)
print('\nmean_ssim_dfield_similar_tissues2:\n', mean_ssim_dfield_similar_tissues2)
print('\nmean_ssim_dfield_similar_patches:\n', mean_ssim_dfield_similar_patches)




INITIAL:
    mean_csf   mean_gm  mean_wm
0  0.842664  0.805065  0.90006

mean_ssim_dfield_mean:
      mean_csf   mean_gm   mean_wm
10   0.883359  0.807923  0.904353
100  0.888534  0.817308  0.909024
200  0.887947  0.817732  0.909262
300  0.888056  0.818064  0.909415
620  0.888074  0.818042  0.909481

mean_ssim_dfield_similar_images:
                mean_csf   mean_gm   mean_wm
SSIM_5         0.902174  0.816281  0.908979
SSIM_10        0.903318  0.819518  0.910730
SSIM_100       0.900513  0.820392  0.911014
DICE_MEAN_5    0.901161  0.815557  0.908746
DICE_MEAN_10   0.903199  0.818977  0.910590
DICE_MEAN_100  0.900946  0.820726  0.911393

mean_ssim_dfield_similar_images_registered:
             mean_csf   mean_gm   mean_wm
AFF_5       0.902095  0.815810  0.908977
BSP_FAST_5  0.902871  0.816563  0.909390

mean_ssim_dfield_similar_tissues:
         mean_csf   mean_gm   mean_wm
3_10    0.902553  0.816275  0.910788
3_100   0.899670  0.816718  0.911102
96_10   0.903537  0.816292  0.910937
96_

In [5]:
ssim_initial = get_initial_metric('ssim')
ssim_dfield_mean = get_result_metric('dfield_mean', 'statistics', 'ssim')
ssim_dfield_similar_images = get_result_metric('dfield_similar_images', 'statistics', 'ssim')
ssim_dfield_similar_images_registered = get_result_metric('dfield_similar_images_registered', 'statistics', 'ssim')
ssim_dfield_similar_tissues = get_result_metric('dfield_similar_tissues', 'statistics', 'ssim')
ssim_dfield_similar_tissues2 = get_result_metric('dfield_similar_tissues_v2', 'statistics', 'ssim')
ssim_dfield_similar_patches = get_result_metric('dfield_similar_patches', 'statistics', 'ssim')


def show_t_statistics(ssim_initial, ssim_res):

    for i in range(0, ssim_res.shape[1]-1):
        t_stat_A, p_val_A = stats.ttest_rel(ssim_res.values[:, i+1], ssim_initial.values)
        print(f'{ssim_res.columns[i+1]}: {t_stat_A} - {p_val_A}')

print('mean_ssim_dfield_mean:')
show_t_statistics(ssim_initial, ssim_dfield_mean)
print('\nmean_ssim_dfield_similar_images:')
show_t_statistics(ssim_initial, ssim_dfield_similar_images)
print('\nmean_ssim_dfield_similar_images_registered:')
show_t_statistics(ssim_initial, ssim_dfield_similar_images_registered)
print('\nmean_ssim_dfield_similar_tissues:')
show_t_statistics(ssim_initial, ssim_dfield_similar_tissues)
print('\nmean_ssim_dfield_similar_tissues:')
show_t_statistics(ssim_initial, ssim_dfield_similar_tissues2)
print('\nmean_ssim_dfield_similar_patches:')
show_t_statistics(ssim_initial, ssim_dfield_similar_patches)



mean_ssim_dfield_mean:
10: 8.362102550260321 - 1.3363966875842007e-12
100: 17.421777418425506 - 2.7087867711173113e-29
200: 18.435886004408605 - 6.651864480164869e-31
300: 18.555186721244134 - 4.33636606504653e-31
620: 18.184831966019317 - 1.6459073931917302e-30

mean_ssim_dfield_similar_images:
SSIM_5: 13.999469730735642 - 1.9008726802368097e-23
SSIM_10: 17.337250358330916 - 3.7105150983865724e-29
SSIM_100: 17.767826841548477 - 7.538757958306391e-30
DICE_MEAN_5: 11.609012475881318 - 5.423631459780721e-19
DICE_MEAN_10: 13.22808758492151 - 4.842939131606832e-22
DICE_MEAN_100: 14.977773885525327 - 3.4823344150429922e-25

mean_ssim_dfield_similar_images_registered:
AFF_5: 11.387218458603916 - 1.449939857348767e-18
BSP_FAST_5: 11.608478421279708 - 5.436457577859906e-19

mean_ssim_dfield_similar_tissues:
3_10: 20.718998570791484 - 2.4668974018216043e-34
3_100: 21.171338359755428 - 5.530073054333703e-35
96_10: 20.684948303639963 - 2.76330990162671e-34
96_100: 21.881389561560933 - 5.527733476

# Compare best methods

In [6]:

def get_mean(approach_name, statistics_folder, metric):
    _metric = pd.read_csv(f'{current_path}/{approach_name}/images/{statistics_folder}/{metric}.csv')
    return _metric.mean().values[1]

approach_name_list = ["Initial", "dfield_similar_patches", "MPGAN_paper_16_seg_loss", "MGAN_paper_16", "StarGAN_V2_sagital"]
metric_list = ["ssim", "dice_csf", "dice_gm", "dice_wm"]
# metric_list = ["ssim"]
for metric in metric_list:
    print(f'{metric} mean'.upper())
    for approach_name in approach_name_list:
            if approach_name == "Initial":
                print(f'{approach_name}:\t\t\t', get_initial_mean(metric))
            elif approach_name == "MPGAN_paper_16_seg_loss":
                print(f'{approach_name} RAW:\t', get_mean(approach_name, "statistics_postprocessed", metric + "_raw"))
                print(f'{approach_name} POST:\t', get_mean(approach_name, "statistics_postprocessed", metric + "_post"))
            elif approach_name == "dfield_similar_patches":
                print(f'{approach_name}:\t\t', get_mean(approach_name, "statistics_postprocessed", metric))
            elif approach_name == "MGAN_paper_16":
                print(f'{approach_name}:\t\t\t', get_mean(approach_name, "statistics_postprocessed", metric))
            elif approach_name == "StarGAN_V2_sagital":
                print(f'{approach_name}:\t\t', get_mean(approach_name, "statistics_postprocessed", metric))
    print('')


SSIM MEAN
Initial:			 0.9463590432890597
dfield_similar_patches:		 0.9580158316164026
MPGAN_paper_16_seg_loss RAW:	 0.9485469389841606
MPGAN_paper_16_seg_loss POST:	 0.9272257148807934
MGAN_paper_16:			 0.9258335220137155
StarGAN_V2_sagital:		 0.9275009487400047

DICE_CSF MEAN
Initial:			 0.8426644
dfield_similar_patches:		 0.922201029920219
MPGAN_paper_16_seg_loss RAW:	 0.906797215690076
MPGAN_paper_16_seg_loss POST:	 0.9009654682925385
MGAN_paper_16:			 0.8922287746772473
StarGAN_V2_sagital:		 0.8242091928120957

DICE_GM MEAN
Initial:			 0.80506486
dfield_similar_patches:		 0.8413915756618774
MPGAN_paper_16_seg_loss RAW:	 0.8137642209791004
MPGAN_paper_16_seg_loss POST:	 0.7496597890349561
MGAN_paper_16:			 0.7271998346811397
StarGAN_V2_sagital:		 0.7522723593925006

DICE_WM MEAN
Initial:			 0.9000597
dfield_similar_patches:		 0.9223467837043023
MPGAN_paper_16_seg_loss RAW:	 0.9100585846967613
MPGAN_paper_16_seg_loss POST:	 0.8801782088922083
MGAN_paper_16:			 0.8711045668868814
Star

### Brain parenchymal fraction

In [7]:
val_images = ["08202"] + pil.get_matching_ids()[621:]

bpf_orig = pd.read_csv(f'C:/Users/Agustin/Master/Master/S4/code/statistics/patients/cross_sectional/whole_brain/features/whole_brains_stats.csv')
bpf_orig = bpf_orig[bpf_orig["patient_id"].isin([int(x) for x in val_images])]
bpf_orig = bpf_orig.reset_index(drop=True)
bpf_dfield_similar = pd.read_csv(f'{current_path}/dfield_similar_patches/images/statistics_postprocessed/bpf.csv')
bpf_MPGAN_raw = pd.read_csv(f'{current_path}/MPGAN_paper_16_seg_loss/images/statistics_postprocessed/bpf_raw.csv')
bpf_starGAN = pd.read_csv(f'{current_path}/StarGAN_V2_sagital/images/statistics_postprocessed/bpf.csv')

bpf_orig["bpf_sim_patches"] = bpf_dfield_similar["96_10"]
bpf_orig["bpf_MPGAN_raw"] = bpf_MPGAN_raw["52"]
bpf_orig["bpf_starGAN"] = bpf_starGAN["BEST"]
bpf_orig["bpf_sim_patches_dif"] = (bpf_orig["bpf_hunt3"] - bpf_orig["bpf_sim_patches"])/(bpf_orig["bpf_hunt3"] - bpf_orig["bpf_hunt4"])
bpf_orig["bpf_MPGAN_raw_dif"] = (bpf_orig["bpf_hunt3"] - bpf_orig["bpf_MPGAN_raw"])/(bpf_orig["bpf_hunt3"] - bpf_orig["bpf_hunt4"])
bpf_orig["bpf_starGAN_dif"] = (bpf_orig["bpf_hunt3"] - bpf_orig["bpf_starGAN"])/(bpf_orig["bpf_hunt3"] - bpf_orig["bpf_hunt4"])
display(bpf_orig.head(20))
print("BPF MEAN:\n", bpf_orig.mean()[1:])
print("\nBPF MAX:\n", bpf_orig.max()[1:])


,patient_id,bpf_hunt3,bpf_hunt4,bpf_sim_patches,bpf_MPGAN_raw,bpf_starGAN,bpf_sim_patches_dif,bpf_MPGAN_raw_dif,bpf_starGAN_dif
0,8202.0,0.786418,0.752760,0.776017,0.753120,0.726654,0.308999,0.989299,1.775611
1,14098.0,0.741002,0.702721,0.729728,0.736311,0.695885,0.294507,0.122553,1.178571
2,14104.0,0.745154,0.712419,0.726263,0.724892,0.692954,0.577091,0.618978,1.594602
3,14128.0,0.761222,0.734833,0.750827,0.748110,0.715115,0.393912,0.496876,1.747245
4,14135.0,0.783011,0.752353,0.772091,0.756886,0.739828,0.356191,0.852141,1.408547
5,14159.0,0.745675,0.720731,0.732272,0.732460,0.695446,0.537342,0.529788,2.013645
6,14180.0,0.737479,0.709757,0.725169,0.734413,0.702920,0.444056,0.110598,1.246610
7,14197.0,0.731210,0.702301,0.722043,0.731534,0.703492,0.317075,-0.011220,0.958781
8,14203.0,0.757322,0.729706,0.741024,0.743056,0.707223,0.590168,0.516577,1.814164
9,14241.0,0.783564,0.752312,0.774461,0.758149,0.730384,0.291265,0.813237,1.701656


BPF MEAN:
 bpf_hunt3              0.748551
bpf_hunt4              0.717533
bpf_sim_patches        0.735342
bpf_MPGAN_raw          0.736028
bpf_starGAN            0.705260
bpf_sim_patches_dif    0.440817
bpf_MPGAN_raw_dif      0.408774
bpf_starGAN_dif        1.448348
dtype: float64

BPF MAX:
 bpf_hunt3              0.786418
bpf_hunt4              0.755226
bpf_sim_patches        0.776017
bpf_MPGAN_raw          0.759183
bpf_starGAN            0.739828
bpf_sim_patches_dif    0.789446
bpf_MPGAN_raw_dif      0.989299
bpf_starGAN_dif        2.712205
dtype: float64
